# Nemotron Leaderboard Engine

Use this notebook as the Kaggle-side competition notebook for the NVIDIA Nemotron Model Reasoning Challenge.

Before running:
- Attach the competition input.
- Attach the Kaggle model `metric/nemotron-3-nano-30b-a3b-bf16/Transformers/default`.
- Set `Settings -> Accelerator -> GPU RTX Pro 6000` first.
- If this competition blocks internet on the current accelerator, attach an offline wheelhouse dataset with the required `.whl` files for Torch and Nemotron runtime dependencies.

What this notebook now does automatically:
- Creates a unique run folder under `/kaggle/working/artifacts/kaggle_runs/<run_id>/`.
- Logs every major stage to `events.jsonl`.
- Writes structured summaries for environment, data discovery, model load, smoke tests, and submission scaffold generation.
- Builds a zipped run bundle in `/kaggle/working` so each Kaggle attempt is exportable.

Run strategy:
1. Run the environment and input inspection cells.
2. Run the competition/model discovery cells.
3. If you attached a wheelhouse dataset, let the notebook detect it.
4. Run the Blackwell compatibility cell. If it upgrades PyTorch, the kernel will restart automatically.
5. After restart, rerun from the top until you reach the runtime dependency cell.
6. Let the notebook install `mamba-ssm` and `causal-conv1d` if needed.
7. Run the tokenizer cell.
8. Run the full model load cell.
9. If the model loads, run the short generation smoke test.
10. Review the exported run bundle paths at the end and sync the outputs back into the repo loop.

In [ ]:
import json
import os
import platform
import shutil
import socket
import time
import traceback
import urllib.request
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import pandas as pd
import torch

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
COMP_DIR = Path('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge')

NOTEBOOK_NAME = 'nemotron_leaderboard_engine'
RUN_ID = os.environ.get('NEMOTRON_RUN_ID') or f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{uuid4().hex[:8]}"
RUN_ARTIFACT_ROOT = WORK_ROOT / 'artifacts' / 'kaggle_runs' / RUN_ID
RUN_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
EVENT_LOG_PATH = RUN_ARTIFACT_ROOT / 'events.jsonl'
SESSION_SUMMARY_PATH = RUN_ARTIFACT_ROOT / 'session_summary.json'
SYNC_HINT_PATH = RUN_ARTIFACT_ROOT / 'sync_hint.txt'

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str), encoding='utf-8')


def write_text(path: Path, payload: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(payload, encoding='utf-8')


def dedupe_paths(paths: list[Path]) -> list[Path]:
    unique: list[Path] = []
    seen: set[str] = set()
    for path in paths:
        try:
            key = str(path.resolve())
        except Exception:
            key = str(path)
        if key in seen:
            continue
        seen.add(key)
        unique.append(path)
    return unique


def discover_wheelhouse_dirs(root: Path) -> list[Path]:
    wheel_counts: dict[Path, int] = {}
    for wheel_path in root.rglob('*.whl'):
        wheel_counts[wheel_path.parent] = wheel_counts.get(wheel_path.parent, 0) + 1
    return sorted(wheel_counts, key=lambda path: (-wheel_counts[path], str(path)))


def wheelhouse_has_package(wheelhouse: Path | None, package_prefixes: tuple[str, ...]) -> bool:
    if wheelhouse is None:
        return False
    normalized_prefixes = tuple(prefix.lower().replace('-', '_') for prefix in package_prefixes)
    for wheel_path in wheelhouse.glob('*.whl'):
        stem = wheel_path.name.lower().replace('-', '_')
        if any(stem.startswith(prefix) for prefix in normalized_prefixes):
            return True
    return False


def log_event(event_type: str, **payload):
    record = {
        'ts': utc_now_iso(),
        'event_type': event_type,
        'run_id': RUN_ID,
        **payload,
    }
    with EVENT_LOG_PATH.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, sort_keys=True, default=str) + '\n')
    return record


SESSION_STATE = {
    'run_id': RUN_ID,
    'notebook_name': NOTEBOOK_NAME,
    'artifact_root': str(RUN_ARTIFACT_ROOT),
    'event_log_path': str(EVENT_LOG_PATH),
    'started_at': utc_now_iso(),
}


def update_session(**payload):
    SESSION_STATE.update(payload)
    write_json(SESSION_SUMMARY_PATH, SESSION_STATE)


write_text(
    SYNC_HINT_PATH,
    '\n'.join([
        f'Run ID: {RUN_ID}',
        f'Artifact root: {RUN_ARTIFACT_ROOT}',
        f'Event log: {EVENT_LOG_PATH}',
        'Persist this run by saving the Kaggle version or exporting the generated zip bundle from /kaggle/working.',
    ]),
)

print('Run ID:', RUN_ID)
print('Artifact root:', RUN_ARTIFACT_ROOT)
print('Torch version:', torch.__version__)
print('Torch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('Input root exists:', INPUT_ROOT.exists())
print('Working root:', WORK_ROOT)

MANUAL_WHEELHOUSE = Path(os.environ['NEMOTRON_WHEELHOUSE_DIR']) if os.environ.get('NEMOTRON_WHEELHOUSE_DIR') else None
KAGGLEHUB_CACHE_ROOT = Path('/root/.cache/kagglehub/datasets')
FALLBACK_WHEELHOUSE_ROOTS = [
    KAGGLEHUB_CACHE_ROOT,
    Path.home() / '.cache' / 'kagglehub' / 'datasets',
    WORK_ROOT / 'offline_packages',
    WORK_ROOT / 'wheelhouse',
]
WHEELHOUSE_SCAN_ROOTS = dedupe_paths([
    root for root in [INPUT_ROOT, MANUAL_WHEELHOUSE, *FALLBACK_WHEELHOUSE_ROOTS]
    if root is not None and root.exists()
])
WHEELHOUSE_DIRS = []
for wheelhouse_root in WHEELHOUSE_SCAN_ROOTS:
    WHEELHOUSE_DIRS.extend(discover_wheelhouse_dirs(wheelhouse_root))
WHEELHOUSE_DIRS = dedupe_paths(WHEELHOUSE_DIRS)
PRIMARY_WHEELHOUSE = WHEELHOUSE_DIRS[0] if WHEELHOUSE_DIRS else None
print('Wheelhouse scan roots:')
for wheelhouse_root in WHEELHOUSE_SCAN_ROOTS[:10]:
    print('-', wheelhouse_root)
print('Wheelhouse directories detected:')
for wheelhouse_dir in WHEELHOUSE_DIRS[:10]:
    print('-', wheelhouse_dir)
print('Primary wheelhouse:', PRIMARY_WHEELHOUSE)

GPU_NAME = None
GPU_CAPABILITY = None
GPU_VRAM_GB = None
BLACKWELL_GPU = False

if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f'GPU {index}: {torch.cuda.get_device_name(index)}')
        print('  total_memory_gb =', round(props.total_memory / 1024**3, 2))

    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_CAPABILITY = torch.cuda.get_device_capability(0)
    GPU_VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    BLACKWELL_GPU = 'blackwell' in GPU_NAME.lower() or GPU_CAPABILITY[0] >= 12
    print('GPU capability:', GPU_CAPABILITY)
    print('Blackwell GPU detected:', BLACKWELL_GPU)
else:
    print('No CUDA GPU detected. Stop here and turn on a GPU before loading the model.')

runtime_summary = {
    'torch_version': torch.__version__,
    'torch_cuda_build': torch.version.cuda,
    'cuda_available': bool(torch.cuda.is_available()),
    'gpu_name': GPU_NAME,
    'gpu_capability': GPU_CAPABILITY,
    'gpu_vram_gb': GPU_VRAM_GB,
    'blackwell_gpu': BLACKWELL_GPU,
    'wheelhouse_scan_roots': [str(path) for path in WHEELHOUSE_SCAN_ROOTS[:10]],
    'wheelhouse_dirs': [str(path) for path in WHEELHOUSE_DIRS[:10]],
    'primary_wheelhouse': str(PRIMARY_WHEELHOUSE) if PRIMARY_WHEELHOUSE is not None else None,
    'python_version': platform.python_version(),
    'hostname': socket.gethostname(),
    'kaggle_kernel_run_type': os.environ.get('KAGGLE_KERNEL_RUN_TYPE'),
}
write_json(RUN_ARTIFACT_ROOT / 'runtime_summary.json', runtime_summary)
update_session(runtime=runtime_summary)
log_event('session_started', runtime=runtime_summary)

In [ ]:
top_level_inputs = sorted(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
print('Top-level /kaggle/input entries:')
for path in top_level_inputs:
    print('-', path)

sample_submission_candidates = sorted(INPUT_ROOT.rglob('sample_submission.csv'))
test_like_candidates = sorted(
    path for path in INPUT_ROOT.rglob('*')
    if path.name.lower() in {'test.csv', 'test.parquet', 'test.jsonl'}
)
train_like_candidates = sorted(
    path for path in INPUT_ROOT.rglob('*')
    if path.name.lower() in {'train.csv', 'train.parquet', 'train.jsonl'}
)

print('\nSample submission candidates:')
for path in sample_submission_candidates[:10]:
    print('-', path)

print('\nTest file candidates:')
for path in test_like_candidates[:10]:
    print('-', path)

print('\nTrain file candidates:')
for path in train_like_candidates[:10]:
    print('-', path)

input_summary = {
    'top_level_inputs': [str(path) for path in top_level_inputs],
    'sample_submission_candidates': [str(path) for path in sample_submission_candidates[:10]],
    'test_candidates': [str(path) for path in test_like_candidates[:10]],
    'train_candidates': [str(path) for path in train_like_candidates[:10]],
}
write_json(RUN_ARTIFACT_ROOT / 'input_inventory.json', input_summary)
update_session(input_inventory=input_summary)
log_event('inputs_discovered', **input_summary)

In [ ]:
def find_model_candidates(search_root: Path) -> list[Path]:
    candidates = []
    for config_path in search_root.rglob('config.json'):
        parent = config_path.parent
        if (parent / 'tokenizer_config.json').exists():
            candidates.append(parent)

    return sorted(
        set(candidates),
        key=lambda path: ('nemotron' not in str(path).lower(), len(path.parts), str(path)),
    )


MODEL_CANDIDATES = find_model_candidates(INPUT_ROOT)
print('Model candidates:')
for index, path in enumerate(MODEL_CANDIDATES):
    print(index, path)

assert MODEL_CANDIDATES, 'No model root detected under /kaggle/input.'
MODEL_DIR = MODEL_CANDIDATES[0]
print('\nSelected MODEL_DIR =', MODEL_DIR)

SAMPLE_SUBMISSION_PATH = sample_submission_candidates[0] if sample_submission_candidates else None
TEST_DATA_PATH = test_like_candidates[0] if test_like_candidates else None
TRAIN_DATA_PATH = train_like_candidates[0] if train_like_candidates else None
print('SAMPLE_SUBMISSION_PATH =', SAMPLE_SUBMISSION_PATH)
print('TEST_DATA_PATH =', TEST_DATA_PATH)
print('TRAIN_DATA_PATH =', TRAIN_DATA_PATH)

model_selection_summary = {
    'model_candidates': [str(path) for path in MODEL_CANDIDATES],
    'selected_model_dir': str(MODEL_DIR),
    'sample_submission_path': str(SAMPLE_SUBMISSION_PATH) if SAMPLE_SUBMISSION_PATH is not None else None,
    'test_data_path': str(TEST_DATA_PATH) if TEST_DATA_PATH is not None else None,
    'train_data_path': str(TRAIN_DATA_PATH) if TRAIN_DATA_PATH is not None else None,
}
write_json(RUN_ARTIFACT_ROOT / 'model_selection.json', model_selection_summary)
update_session(model_selection=model_selection_summary)
log_event('model_selected', **model_selection_summary)

In [ ]:
print('Competition files:')
competition_files = []
if COMP_DIR.exists():
    for path in sorted(COMP_DIR.rglob('*')):
        if path.is_file():
            competition_files.append(str(path))
            print('-', path)
else:
    print('Competition directory not found:', COMP_DIR)

TEST_COLUMNS = []
TRAIN_COLUMNS = []
test_preview_path = None
train_preview_path = None

if TEST_DATA_PATH is not None:
    test_preview = pd.read_csv(TEST_DATA_PATH, nrows=5)
    TEST_COLUMNS = list(test_preview.columns)
    print('\ntest.csv columns:', TEST_COLUMNS)
    display(test_preview)
    test_preview_path = RUN_ARTIFACT_ROOT / 'test_preview.csv'
    test_preview.to_csv(test_preview_path, index=False)
else:
    print('TEST_DATA_PATH is not available.')

if TRAIN_DATA_PATH is not None:
    train_preview = pd.read_csv(TRAIN_DATA_PATH, nrows=5)
    TRAIN_COLUMNS = list(train_preview.columns)
    print('\ntrain.csv columns:', TRAIN_COLUMNS)
    display(train_preview)
    train_preview_path = RUN_ARTIFACT_ROOT / 'train_preview.csv'
    train_preview.to_csv(train_preview_path, index=False)
else:
    print('TRAIN_DATA_PATH is not available.')

competition_summary = {
    'competition_files': competition_files,
    'test_columns': TEST_COLUMNS,
    'train_columns': TRAIN_COLUMNS,
    'test_preview_path': str(test_preview_path) if test_preview_path is not None else None,
    'train_preview_path': str(train_preview_path) if train_preview_path is not None else None,
}
write_json(RUN_ARTIFACT_ROOT / 'competition_summary.json', competition_summary)
update_session(competition=competition_summary)
log_event('competition_inspected', **competition_summary)

## Heuristic Baseline

This baseline does not require the Nemotron model. It routes prompts into the known competition families and applies symbolic heuristics for the easier families.

Use it to get a local exact-match score on `train.csv` and to emit a first real `submission.csv` from `test.csv` while the model runtime path is still stabilizing.

In [ ]:
from collections import Counter
from decimal import Decimal, ROUND_HALF_UP
from enum import Enum
import re

RUN_HEURISTIC_BASELINE = True

class CompetitionFamily(str, Enum):
    BIT_MANIPULATION = 'bit_manipulation'
    GRAVITY_CONSTANT = 'gravity_constant'
    UNIT_CONVERSION = 'unit_conversion'
    TEXT_DECRYPTION = 'text_decryption'
    NUMERAL_SYSTEM = 'numeral_system'
    EQUATION_TRANSFORM = 'equation_transform'
    UNKNOWN = 'unknown'

FAMILY_PATTERNS = [
    ('a secret bit manipulation rule transforms 8-bit binary numbers', CompetitionFamily.BIT_MANIPULATION),
    ('the gravitational constant has been secretly changed', CompetitionFamily.GRAVITY_CONSTANT),
    ('a secret unit conversion is applied to measurements', CompetitionFamily.UNIT_CONVERSION),
    ('secret encryption rules are used on text', CompetitionFamily.TEXT_DECRYPTION),
    ('numbers are secretly converted into a different numeral system', CompetitionFamily.NUMERAL_SYSTEM),
    ('a secret set of transformation rules is applied to equations', CompetitionFamily.EQUATION_TRANSFORM),
]

ROMAN_NUMERALS = [
    (1000, 'M'), (900, 'CM'), (500, 'D'), (400, 'CD'),
    (100, 'C'), (90, 'XC'), (50, 'L'), (40, 'XL'),
    (10, 'X'), (9, 'IX'), (5, 'V'), (4, 'IV'), (1, 'I'),
]

def classify_prompt_family(prompt: str) -> CompetitionFamily:
    lowered = prompt.lower()
    for pattern, family in FAMILY_PATTERNS:
        if pattern in lowered:
            return family
    return CompetitionFamily.UNKNOWN

def format_decimal(value: Decimal, keep_two_places: bool) -> str:
    quantized = value.quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
    text = format(quantized, 'f')
    if keep_two_places:
        if '.' not in text:
            text = f'{text}.00'
        decimals = text.split('.', 1)[1]
        if len(decimals) == 1:
            text = f'{text}0'
        return text
    if '.' not in text:
        return f'{text}.0'
    whole, decimals = text.split('.', 1)
    if decimals == '00':
        return f'{whole}.0'
    if decimals.endswith('0'):
        return f'{whole}.{decimals[0]}'
    return text

def solve_roman_numeral(prompt: str) -> str | None:
    match = re.search(r'Now, write the number ([0-9]+) in the Wonderland numeral system\.', prompt)
    if match is None:
        return None
    value = int(match.group(1))
    pieces = []
    for amount, symbol in ROMAN_NUMERALS:
        while value >= amount:
            value -= amount
            pieces.append(symbol)
    return ''.join(pieces)

def solve_gravity_constant(prompt: str) -> str | None:
    examples = [(Decimal(t), Decimal(d)) for t, d in re.findall(r'For t = ([0-9.]+)s, distance = ([0-9.]+) m', prompt)]
    query_match = re.search(r'for t = ([0-9.]+)s given d = 0.5\*g\*t\^2', prompt)
    if not examples or query_match is None:
        return None
    lower_bound = None
    upper_bound = None
    for time_value, distance_value in examples:
        time_squared = time_value * time_value
        local_lower = (Decimal('2') * (distance_value - Decimal('0.005'))) / time_squared
        local_upper = (Decimal('2') * (distance_value + Decimal('0.005'))) / time_squared
        lower_bound = local_lower if lower_bound is None else max(lower_bound, local_lower)
        upper_bound = local_upper if upper_bound is None else min(upper_bound, local_upper)
    if lower_bound is None or upper_bound is None or not lower_bound < upper_bound:
        gravity_constant = sum((Decimal('2') * distance_value) / (time_value * time_value) for time_value, distance_value in examples) / Decimal(len(examples))
    else:
        gravity_constant = (lower_bound + upper_bound) / Decimal('2')
    query_time = Decimal(query_match.group(1))
    predicted_distance = Decimal('0.5') * gravity_constant * query_time * query_time
    return format_decimal(predicted_distance, keep_two_places=False)

def solve_unit_conversion(prompt: str) -> str | None:
    examples = [(Decimal(left), Decimal(right)) for left, right in re.findall(r'([0-9.]+) m becomes ([0-9.]+)', prompt)]
    query_match = re.search(r'Now, convert the following measurement: ([0-9.]+) m', prompt)
    if not examples or query_match is None:
        return None
    lower_bound = None
    upper_bound = None
    for source_value, target_value in examples:
        local_lower = (target_value - Decimal('0.005')) / source_value
        local_upper = (target_value + Decimal('0.005')) / source_value
        lower_bound = local_lower if lower_bound is None else max(lower_bound, local_lower)
        upper_bound = local_upper if upper_bound is None else min(upper_bound, local_upper)
    if lower_bound is None or upper_bound is None or not lower_bound < upper_bound:
        multiplier = sum(target_value / source_value for source_value, target_value in examples) / Decimal(len(examples))
    else:
        multiplier = (lower_bound + upper_bound) / Decimal('2')
    query_value = Decimal(query_match.group(1))
    predicted_value = multiplier * query_value
    return format_decimal(predicted_value, keep_two_places=True)

def solve_text_decryption(prompt: str) -> str | None:
    parts = prompt.split('Now, decrypt the following text:')
    if len(parts) != 2:
        return None
    examples_text, query_text = parts
    example_pairs = re.findall(r'([^\n]+?) -> ([^\n]+)', examples_text)
    source_to_target = {}
    target_to_source = {}
    for source, target in example_pairs:
        if len(source) != len(target):
            return None
        for source_char, target_char in zip(source, target):
            if source_to_target.get(source_char, target_char) != target_char:
                return None
            if target_to_source.get(target_char, source_char) != source_char:
                return None
            source_to_target[source_char] = target_char
            target_to_source[target_char] = source_char
    query = query_text.strip()
    if any(char != ' ' and char not in source_to_target for char in query):
        return None
    return ''.join(source_to_target.get(char, char) for char in query)

def solve_gf2(matrix: list[list[int]], vector: list[int]) -> list[int] | None:
    rows = [row[:] + [value] for row, value in zip(matrix, vector)]
    row_count = len(rows)
    column_count = len(matrix[0]) if matrix else 0
    pivot_columns = []
    pivot_row = 0
    for column in range(column_count):
        pivot = None
        for candidate in range(pivot_row, row_count):
            if rows[candidate][column] == 1:
                pivot = candidate
                break
        if pivot is None:
            continue
        rows[pivot_row], rows[pivot] = rows[pivot], rows[pivot_row]
        pivot_columns.append(column)
        for candidate in range(row_count):
            if candidate == pivot_row or rows[candidate][column] == 0:
                continue
            for index in range(column, column_count + 1):
                rows[candidate][index] = (rows[candidate][index] + rows[pivot_row][index]) % 2
        pivot_row += 1
        if pivot_row == row_count:
            break
    for candidate in range(row_count):
        if any(rows[candidate][column] == 1 for column in range(column_count)) is False and rows[candidate][column_count] == 1:
            return None
    solution = [0] * column_count
    for row_index, column in enumerate(pivot_columns):
        solution[column] = rows[row_index][column_count]
    return solution

def apply_gf2_solution(solution: list[int], bits: list[int]) -> int:
    value = 0
    for coefficient, bit in zip(solution, bits):
        value = (value + (coefficient * bit)) % 2
    return value

def solve_bit_manipulation_affine(prompt: str) -> str | None:
    examples = [(source, target) for source, target in re.findall(r'([01]{8}) -> ([01]{8})', prompt)]
    query_match = re.search(r'output for: ([01]{8})', prompt)
    if not examples or query_match is None:
        return None
    matrix = []
    targets = [[] for _ in range(8)]
    for source, target in examples:
        source_bits = [int(char) for char in source] + [1]
        matrix.append(source_bits)
        for index, target_char in enumerate(target):
            targets[index].append(int(target_char))
    coefficients = []
    for target_bits in targets:
        solution = solve_gf2(matrix, target_bits)
        if solution is None:
            return None
        coefficients.append(solution)
    for source, target in examples:
        source_bits = [int(char) for char in source] + [1]
        predicted = ''.join(str(apply_gf2_solution(solution, source_bits)) for solution in coefficients)
        if predicted != target:
            return None
    query_bits = [int(char) for char in query_match.group(1)] + [1]
    return ''.join(str(apply_gf2_solution(solution, query_bits)) for solution in coefficients)

def solve_prompt_heuristic(prompt: str) -> tuple[str | None, CompetitionFamily, str]:
    family = classify_prompt_family(prompt)
    if family == CompetitionFamily.NUMERAL_SYSTEM:
        return solve_roman_numeral(prompt), family, 'roman_symbolic'
    if family == CompetitionFamily.GRAVITY_CONSTANT:
        return solve_gravity_constant(prompt), family, 'gravity_interval'
    if family == CompetitionFamily.UNIT_CONVERSION:
        return solve_unit_conversion(prompt), family, 'unit_interval'
    if family == CompetitionFamily.TEXT_DECRYPTION:
        return solve_text_decryption(prompt), family, 'char_substitution'
    if family == CompetitionFamily.BIT_MANIPULATION:
        return solve_bit_manipulation_affine(prompt), family, 'gf2_affine'
    return None, family, 'unsupported'

HEURISTIC_BASELINE_SUMMARY = None
HEURISTIC_SUBMISSION_PATH = None

if RUN_HEURISTIC_BASELINE and TRAIN_DATA_PATH is not None and TEST_DATA_PATH is not None:
    train_df = pd.read_csv(TRAIN_DATA_PATH)
    test_df = pd.read_csv(TEST_DATA_PATH)

    train_predictions = []
    family_correct = Counter()
    family_total = Counter()
    family_solved = Counter()
    total_correct = 0

    for record in train_df.to_dict(orient='records'):
        prediction, family, solver = solve_prompt_heuristic(record['prompt'])
        matched = prediction == record['answer']
        family_total[family.value] += 1
        if prediction is not None:
            family_solved[family.value] += 1
        if matched:
            family_correct[family.value] += 1
            total_correct += 1
        train_predictions.append({
            'id': record['id'],
            'family': family.value,
            'solver': solver,
            'prediction': prediction or '',
            'answer': record['answer'],
            'matched': matched,
        })

    family_accuracy = {}
    for family_name, total in sorted(family_total.items()):
        family_accuracy[family_name] = {
            'total': int(total),
            'correct': int(family_correct[family_name]),
            'solved': int(family_solved[family_name]),
            'accuracy': float(family_correct[family_name] / total if total else 0.0),
            'solved_rate': float(family_solved[family_name] / total if total else 0.0),
        }

    heuristic_train_predictions_df = pd.DataFrame(train_predictions)
    heuristic_train_predictions_path = RUN_ARTIFACT_ROOT / 'heuristic_train_predictions.csv'
    heuristic_train_predictions_df.to_csv(heuristic_train_predictions_path, index=False)

    test_predictions = []
    for record in test_df.to_dict(orient='records'):
        prediction, family, solver = solve_prompt_heuristic(record['prompt'])
        test_predictions.append({
            'id': record['id'],
            'answer': prediction or '',
            'family': family.value,
            'solver': solver,
        })

    heuristic_test_predictions_df = pd.DataFrame(test_predictions)
    heuristic_test_predictions_path = RUN_ARTIFACT_ROOT / 'heuristic_test_predictions.csv'
    heuristic_test_predictions_df.to_csv(heuristic_test_predictions_path, index=False)

    heuristic_submission_df = heuristic_test_predictions_df[['id', 'answer']].copy()
    HEURISTIC_SUBMISSION_PATH = WORK_ROOT / 'heuristic_submission.csv'
    heuristic_submission_df.to_csv(HEURISTIC_SUBMISSION_PATH, index=False)
    heuristic_submission_artifact_path = RUN_ARTIFACT_ROOT / 'heuristic_submission.csv'
    heuristic_submission_df.to_csv(heuristic_submission_artifact_path, index=False)

    HEURISTIC_BASELINE_SUMMARY = {
        'train_row_count': int(len(train_df)),
        'test_row_count': int(len(test_df)),
        'overall_accuracy': float(total_correct / len(train_df) if len(train_df) else 0.0),
        'family_accuracy': family_accuracy,
        'train_predictions_path': str(heuristic_train_predictions_path),
        'test_predictions_path': str(heuristic_test_predictions_path),
        'submission_path': str(HEURISTIC_SUBMISSION_PATH),
        'artifact_submission_path': str(heuristic_submission_artifact_path),
    }
    write_json(RUN_ARTIFACT_ROOT / 'heuristic_baseline_summary.json', HEURISTIC_BASELINE_SUMMARY)
    update_session(heuristic_baseline=HEURISTIC_BASELINE_SUMMARY)
    log_event('heuristic_baseline_completed', summary=HEURISTIC_BASELINE_SUMMARY)

    print('Heuristic baseline accuracy on train.csv:', round(HEURISTIC_BASELINE_SUMMARY['overall_accuracy'], 4))
    for family_name, stats in HEURISTIC_BASELINE_SUMMARY['family_accuracy'].items():
        print(f"{family_name}: accuracy={stats['accuracy']:.4f} solved_rate={stats['solved_rate']:.4f} correct={stats['correct']} total={stats['total']}")
    print('Heuristic submission path:', HEURISTIC_SUBMISSION_PATH)
else:
    print('Skipping heuristic baseline because train/test competition inputs are not both available.')

## Blackwell Compatibility Fix

Kaggle's default PyTorch build may not support the Blackwell GPU architecture yet.

This cell will:
- detect that mismatch,
- install a CUDA 12.8-compatible PyTorch build if needed,
- write a marker file into `/kaggle/working`,
- and restart the kernel automatically.

If the kernel restarts, simply rerun from the top.

In [ ]:
import subprocess
import sys

TORCH_UPGRADE_MARKER = WORK_ROOT / '.torch_blackwell_ready'

def parse_cuda_version(raw: str | None) -> tuple[int, int]:
    if not raw:
        return (0, 0)
    parts = raw.split('.')
    major = int(parts[0]) if parts else 0
    minor = int(parts[1]) if len(parts) > 1 else 0
    return (major, minor)


def install_blackwell_torch() -> None:
    local_torch_wheelhouse = next(
        (
            path for path in WHEELHOUSE_DIRS
            if wheelhouse_has_package(path, ('torch', 'torchvision', 'torchaudio'))
        ),
        None,
    )
    connectivity_status = pypi_connectivity_check()
    online_available = all(item.get('ok') for item in connectivity_status.values())
    log_event(
        'torch_install_environment',
        connectivity=connectivity_status,
        local_torch_wheelhouse=str(local_torch_wheelhouse) if local_torch_wheelhouse is not None else None,
    )

    attempts = []
    if local_torch_wheelhouse is not None:
        attempts.append(
            (
                'offline-wheelhouse',
                [
                    sys.executable,
                    '-m',
                    'pip',
                    'install',
                    '--quiet',
                    '--upgrade',
                    '--no-index',
                    '--find-links',
                    str(local_torch_wheelhouse),
                    'torch',
                    'torchvision',
                    'torchaudio',
                ],
            )
        )

    if online_available:
        attempts.extend([
            (
                'stable-cu128',
                [
                    sys.executable,
                    '-m',
                    'pip',
                    'install',
                    '--quiet',
                    '--upgrade',
                    'torch',
                    'torchvision',
                    'torchaudio',
                    '--index-url',
                    'https://download.pytorch.org/whl/cu128',
                ],
            ),
            (
                'nightly-cu128',
                [
                    sys.executable,
                    '-m',
                    'pip',
                    'install',
                    '--quiet',
                    '--upgrade',
                    '--pre',
                    'torch',
                    'torchvision',
                    'torchaudio',
                    '--index-url',
                    'https://download.pytorch.org/whl/nightly/cu128',
                ],
            ),
        ])

    if not attempts:
        raise RuntimeError(
            'Need a Blackwell-compatible torch wheelhouse, but no offline wheelhouse was attached and internet is unavailable on this Kaggle notebook.'
        )

    last_error = None
    for label, command in attempts:
        print(f'Trying torch install target: {label}')
        log_event('torch_install_attempt', target=label)
        try:
            subprocess.check_call(command)
            log_event('torch_install_succeeded', target=label)
            print(f'Succeeded with {label}')
            return
        except subprocess.CalledProcessError as exc:
            last_error = exc
            log_event('torch_install_failed', target=label, return_code=exc.returncode)
            print(f'Failed with {label}: return code {exc.returncode}')

    raise RuntimeError(
        'Unable to install a Blackwell-compatible torch build. Attach an offline wheelhouse dataset or use a session where internet is available for dependency setup.'
    ) from last_error


current_cuda_build = parse_cuda_version(torch.version.cuda)
needs_blackwell_upgrade = bool(torch.cuda.is_available() and BLACKWELL_GPU and current_cuda_build < (12, 8))

print('Current CUDA build:', current_cuda_build)
print('Needs Blackwell torch upgrade:', needs_blackwell_upgrade)
print('Upgrade marker exists:', TORCH_UPGRADE_MARKER.exists())
log_event(
    'torch_compatibility_checked',
    current_cuda_build=current_cuda_build,
    needs_blackwell_upgrade=needs_blackwell_upgrade,
    upgrade_marker_exists=TORCH_UPGRADE_MARKER.exists(),
)

if needs_blackwell_upgrade and not TORCH_UPGRADE_MARKER.exists():
    print('Installing a Blackwell-compatible torch build...')
    install_blackwell_torch()
    TORCH_UPGRADE_MARKER.write_text('upgraded\n', encoding='utf-8')
    update_session(torch_upgrade={'performed': True, 'marker_path': str(TORCH_UPGRADE_MARKER), 'target_cuda': 'cu128'})
    print('Torch upgrade complete. Restarting kernel now. After restart, rerun from the top.')
    log_event('torch_upgrade_completed', marker_path=str(TORCH_UPGRADE_MARKER))
    os.kill(os.getpid(), 9)
elif needs_blackwell_upgrade:
    update_session(torch_upgrade={'performed': True, 'marker_path': str(TORCH_UPGRADE_MARKER), 'target_cuda': 'cu128'})
    log_event('torch_upgrade_reused', marker_path=str(TORCH_UPGRADE_MARKER))
    print('Blackwell upgrade marker found. Continue with the next cells.')
else:
    update_session(torch_upgrade={'performed': False, 'current_cuda_build': current_cuda_build})
    log_event('torch_upgrade_not_required', current_cuda_build=current_cuda_build)
    print('Current torch build is acceptable for this accelerator. Continue with the next cells.')

## Install Nemotron Runtime Dependencies

The Nemotron hybrid model code imports `mamba-ssm` and `causal-conv1d`. Kaggle does not include these by default.

Keep internet on for this step. If installation fails, stop and capture the exact error.

In [ ]:
import importlib.util
import subprocess

RUNTIME_DEPS_MARKER = WORK_ROOT / '.nemotron_runtime_deps_ready'

def collect_runtime_dep_status() -> dict:
    status = {
        'mamba_ssm_spec': importlib.util.find_spec('mamba_ssm') is not None,
        'causal_conv1d_spec': importlib.util.find_spec('causal_conv1d') is not None,
    }

    try:
        import mamba_ssm  # noqa: F401
        from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn  # noqa: F401
        status['mamba_import_ok'] = True
    except Exception as exc:
        status['mamba_import_ok'] = False
        status['mamba_error_type'] = type(exc).__name__
        status['mamba_error_message'] = str(exc)

    try:
        import causal_conv1d  # noqa: F401
        from causal_conv1d import causal_conv1d_fn  # noqa: F401
        status['causal_conv1d_import_ok'] = True
    except Exception as exc:
        status['causal_conv1d_import_ok'] = False
        status['causal_conv1d_error_type'] = type(exc).__name__
        status['causal_conv1d_error_message'] = str(exc)

    status['ready'] = bool(
        status.get('mamba_import_ok') and status.get('causal_conv1d_import_ok')
    )
    return status


runtime_dep_status = collect_runtime_dep_status()
print('Nemotron runtime dependency status:', runtime_dep_status)
write_json(RUN_ARTIFACT_ROOT / 'runtime_dependency_status_before.json', runtime_dep_status)
log_event('runtime_dependencies_checked', status=runtime_dep_status)
MODEL_RUNTIME_READY = runtime_dep_status['ready']

if not MODEL_RUNTIME_READY:
    local_runtime_wheelhouse = next(
        (
            path for path in WHEELHOUSE_DIRS
            if wheelhouse_has_package(path, ('mamba_ssm', 'causal_conv1d', 'ninja', 'packaging'))
        ),
        PRIMARY_WHEELHOUSE,
    )
    if local_runtime_wheelhouse is not None:
        install_command = [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--quiet',
            '--no-index',
            '--find-links',
            str(local_runtime_wheelhouse),
            '--ignore-installed',
            'mamba-ssm',
            'causal-conv1d',
            'ninja',
            'packaging',
        ]
        print('Installing Nemotron runtime deps from:', local_runtime_wheelhouse)
        log_event('runtime_dependency_install_attempt', wheelhouse=str(local_runtime_wheelhouse))
        try:
            subprocess.check_call(install_command)
            importlib.invalidate_caches()
            runtime_dep_status = collect_runtime_dep_status()
            MODEL_RUNTIME_READY = runtime_dep_status['ready']
        except subprocess.CalledProcessError as exc:
            error_payload = {
                'stage': 'runtime_dependency_install',
                'status': runtime_dep_status,
                'wheelhouse': str(local_runtime_wheelhouse),
                'return_code': exc.returncode,
            }
            write_json(RUN_ARTIFACT_ROOT / 'runtime_dependency_error.json', error_payload)
            update_session(last_error=error_payload, runtime_dependencies=runtime_dep_status)
            log_event('runtime_dependency_install_failed', **error_payload)
    if not MODEL_RUNTIME_READY:
        error_payload = {
            'stage': 'runtime_dependency_check',
            'status': runtime_dep_status,
            'wheelhouse_dirs': [str(path) for path in WHEELHOUSE_DIRS[:10]],
            'hint': 'Attach or prefetch an offline wheelhouse that contains mamba-ssm and causal-conv1d if you want the full Nemotron model path.',
        }
        write_json(RUN_ARTIFACT_ROOT / 'runtime_dependency_error.json', error_payload)
        update_session(last_error=error_payload, runtime_dependencies=runtime_dep_status)

write_json(RUN_ARTIFACT_ROOT / 'runtime_dependency_status_after.json', runtime_dep_status)
update_session(runtime_dependencies=runtime_dep_status, model_runtime_ready=MODEL_RUNTIME_READY)
if MODEL_RUNTIME_READY:
    log_event('runtime_dependencies_ready', status=runtime_dep_status)
    RUNTIME_DEPS_MARKER.write_text('ready\n', encoding='utf-8')
    print('Nemotron runtime dependencies are ready.')
else:
    if RUNTIME_DEPS_MARKER.exists():
        RUNTIME_DEPS_MARKER.unlink()
    log_event('runtime_dependencies_unavailable', status=runtime_dep_status)
    print('Nemotron runtime dependencies are unavailable. The notebook will not attempt installs. Model load cells will be skipped; heuristic baseline output remains usable.')

## Load The Tokenizer First

This should be cheap. If this fails, the attached model path is wrong or the Kaggle model input is incomplete.

In [ ]:
assert 'MODEL_DIR' in globals(), 'Run the model discovery cell first.'

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
tokenizer_summary = {
    'model_dir': str(MODEL_DIR),
    'tokenizer_class': tokenizer.__class__.__name__,
    'vocab_size': getattr(tokenizer, 'vocab_size', 'unknown'),
}
write_json(RUN_ARTIFACT_ROOT / 'tokenizer_summary.json', tokenizer_summary)
update_session(tokenizer=tokenizer_summary)
log_event('tokenizer_loaded', **tokenizer_summary)
print('Tokenizer loaded from:', MODEL_DIR)
print('Tokenizer class:', tokenizer.__class__.__name__)
print('Tokenizer vocab size:', getattr(tokenizer, 'vocab_size', 'unknown'))

## Load The Model

This is the expensive step. If it fails with OOM, stop the session and do not keep retrying blindly.

In [ ]:
if not RUNTIME_DEPS_MARKER.exists():
    model_load_summary = {
        'status': 'skipped',
        'reason': 'runtime_dependencies_unavailable',
    }
    write_json(RUN_ARTIFACT_ROOT / 'model_load.json', model_load_summary)
    update_session(model_load=model_load_summary)
    log_event('model_load_skipped', **model_load_summary)
    print('Skipping model load because Nemotron runtime dependencies are unavailable.')
else:
    assert 'MODEL_DIR' in globals(), 'Run the model discovery cell first.'
    assert 'tokenizer' in globals(), 'Run the tokenizer cell first.'

    TORCH_UPGRADE_MARKER = WORK_ROOT / '.torch_blackwell_ready'
    current_cuda_build = parse_cuda_version(torch.version.cuda)
    assert not (torch.cuda.is_available() and BLACKWELL_GPU and current_cuda_build < (12, 8) and not TORCH_UPGRADE_MARKER.exists()), (
        'Run the Blackwell compatibility cell and let the kernel restart before loading the model.'
    )

    from transformers import AutoModelForCausalLM

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model_load_started = time.perf_counter()

    try:
        model = AutoModelForCausalLM.from_pretrained(
            str(MODEL_DIR),
            torch_dtype=dtype,
            trust_remote_code=True,
            device_map='auto',
            low_cpu_mem_usage=True,
        )
        model.eval()
    except Exception as exc:
        error_payload = {
            'stage': 'model_load',
            'error_type': type(exc).__name__,
            'error_message': str(exc),
            'traceback': traceback.format_exc(),
        }
        write_json(RUN_ARTIFACT_ROOT / 'model_load_error.json', error_payload)
        update_session(last_error=error_payload, model_load={'status': 'failed'})
        log_event('model_load_failed', error_type=error_payload['error_type'], error_message=error_payload['error_message'])
        raise

    def model_input_device(model):
        if hasattr(model, 'device'):
            return model.device
        return next(iter(model.parameters())).device

    model_load_summary = {
        'status': 'loaded',
        'dtype': str(dtype),
        'input_device': str(model_input_device(model)),
        'device_map': getattr(model, 'hf_device_map', 'single-device'),
        'latency_ms': int((time.perf_counter() - model_load_started) * 1000),
    }
    write_json(RUN_ARTIFACT_ROOT / 'model_load.json', model_load_summary)
    update_session(model_load=model_load_summary)
    log_event('model_loaded', **model_load_summary)
    print('Model loaded successfully.')
    print('Input device:', model_input_device(model))
    print('Device map:', getattr(model, 'hf_device_map', 'single-device'))

In [ ]:
if 'model' not in globals():
    smoke_test_summary = {
        'status': 'skipped',
        'reason': 'model_not_loaded',
    }
    write_json(RUN_ARTIFACT_ROOT / 'smoke_test_no_think.json', smoke_test_summary)
    update_session(smoke_test_no_think=smoke_test_summary)
    log_event('smoke_test_skipped', **smoke_test_summary)
    print('Skipping no-thinking smoke test because the model was not loaded.')
else:
    def tokenize_messages(messages, enable_thinking=None):
        kwargs = {
            'tokenize': True,
            'add_generation_prompt': True,
            'return_tensors': 'pt',
        }
        if enable_thinking is not None:
            try:
                return tokenizer.apply_chat_template(messages, enable_thinking=enable_thinking, **kwargs)
            except TypeError:
                print('Tokenizer does not expose enable_thinking; retrying without it.')
        return tokenizer.apply_chat_template(messages, **kwargs)

    messages = [
        {
            'role': 'user',
            'content': 'Solve briefly: what is 17 * 19? Return only the final answer.',
        }
    ]

    inputs = tokenize_messages(messages, enable_thinking=False).to(model_input_device(model))
    generation_started = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=32,
            do_sample=False,
            num_beams=1,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs.shape[-1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    smoke_test_summary = {
        'mode': 'no_think',
        'prompt': messages[0]['content'],
        'generated_text': generated_text,
        'latency_ms': int((time.perf_counter() - generation_started) * 1000),
        'max_new_tokens': 32,
    }
    write_json(RUN_ARTIFACT_ROOT / 'smoke_test_no_think.json', smoke_test_summary)
    update_session(smoke_test_no_think=smoke_test_summary)
    log_event('smoke_test_completed', **smoke_test_summary)
    print(generated_text)

In [ ]:
if 'model' not in globals():
    thinking_summary = {
        'status': 'skipped',
        'reason': 'model_not_loaded',
    }
    write_json(RUN_ARTIFACT_ROOT / 'smoke_test_think.json', thinking_summary)
    update_session(smoke_test_think=thinking_summary)
    log_event('thinking_test_skipped', **thinking_summary)
    print('Skipping thinking test because the model was not loaded.')
else:
    RUN_THINKING_TEST = False

    if RUN_THINKING_TEST:
        thinking_started = time.perf_counter()
        thinking_inputs = tokenize_messages(messages, enable_thinking=True).to(model_input_device(model))
        with torch.no_grad():
            thinking_outputs = model.generate(
                thinking_inputs,
                max_new_tokens=128,
                do_sample=True,
                temperature=1.0,
                top_p=1.0,
                eos_token_id=tokenizer.eos_token_id,
            )
        generated_tokens = thinking_outputs[0][thinking_inputs.shape[-1]:]
        thinking_text = tokenizer.decode(generated_tokens, skip_special_tokens=False)
        thinking_summary = {
            'mode': 'think',
            'prompt': messages[0]['content'],
            'generated_text': thinking_text,
            'latency_ms': int((time.perf_counter() - thinking_started) * 1000),
            'max_new_tokens': 128,
        }
        write_json(RUN_ARTIFACT_ROOT / 'smoke_test_think.json', thinking_summary)
        update_session(smoke_test_think=thinking_summary)
        log_event('thinking_test_completed', **thinking_summary)
        print(thinking_text)
    else:
        log_event('thinking_test_skipped')
        print('Set RUN_THINKING_TEST = True only after the short no-thinking smoke test succeeds.')

In [ ]:
submission_summary = {
    'used_sample_submission': False,
    'submission_path': None,
    'artifact_submission_path': None,
    'columns': None,
}

if HEURISTIC_SUBMISSION_PATH is not None:
    heuristic_submission = pd.read_csv(HEURISTIC_SUBMISSION_PATH)
    submission_path = WORK_ROOT / 'submission.csv'
    artifact_submission_path = RUN_ARTIFACT_ROOT / 'submission.csv'
    heuristic_submission.to_csv(submission_path, index=False)
    heuristic_submission.to_csv(artifact_submission_path, index=False)
    submission_summary.update({
        'used_sample_submission': False,
        'submission_path': str(submission_path),
        'artifact_submission_path': str(artifact_submission_path),
        'columns': list(heuristic_submission.columns),
        'source': 'heuristic_baseline',
    })
    print('Using heuristic submission generated from competition test.csv:', submission_path)
    display(heuristic_submission.head())
elif SAMPLE_SUBMISSION_PATH is not None:
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    print('Sample submission shape:', sample_submission.shape)
    print('Columns:', list(sample_submission.columns))
    display(sample_submission.head())

    prediction_columns = [
        column for column in sample_submission.columns
        if column.lower() not in {'id', 'index'}
    ]

    if prediction_columns:
        submission = sample_submission.copy()
        submission[prediction_columns[0]] = ''
        submission_path = WORK_ROOT / 'submission.csv'
        artifact_submission_path = RUN_ARTIFACT_ROOT / 'submission.csv'
        submission.to_csv(submission_path, index=False)
        submission.to_csv(artifact_submission_path, index=False)
        submission_summary.update({
            'used_sample_submission': True,
            'submission_path': str(submission_path),
            'artifact_submission_path': str(artifact_submission_path),
            'columns': list(submission.columns),
            'source': 'sample_submission_template',
        })
        print('Draft submission scaffold written to:', submission_path)
        display(submission.head())
    else:
        print('Could not infer the prediction column from the sample submission.')
elif TEST_DATA_PATH is not None and TEST_COLUMNS:
    test_id_col = next((column for column in TEST_COLUMNS if column.lower() in {'id', 'index'}), TEST_COLUMNS[0])
    test_ids = pd.read_csv(TEST_DATA_PATH, usecols=[test_id_col])
    placeholder_submission = pd.DataFrame({test_id_col: test_ids[test_id_col], 'answer': ''})
    placeholder_path = WORK_ROOT / 'submission_placeholder.csv'
    artifact_placeholder_path = RUN_ARTIFACT_ROOT / 'submission_placeholder.csv'
    placeholder_submission.to_csv(placeholder_path, index=False)
    placeholder_submission.to_csv(artifact_placeholder_path, index=False)
    submission_summary.update({
        'used_sample_submission': False,
        'submission_path': str(placeholder_path),
        'artifact_submission_path': str(artifact_placeholder_path),
        'columns': list(placeholder_submission.columns),
        'source': 'placeholder_template',
    })
    print('No sample submission was attached. Wrote a provisional placeholder to:', placeholder_path)
    display(placeholder_submission.head())
    print('Replace the provisional answer column name after the official submission schema is confirmed.')
else:
    print('Neither sample submission nor test columns are available.')

write_json(RUN_ARTIFACT_ROOT / 'submission_summary.json', submission_summary)
update_session(submission=submission_summary, completed_at=utc_now_iso())
log_event('submission_scaffold_written', **submission_summary)

bundle_base = WORK_ROOT / f'kaggle_run_{RUN_ID}'
bundle_path = Path(shutil.make_archive(str(bundle_base), 'zip', root_dir=RUN_ARTIFACT_ROOT.parent, base_dir=RUN_ARTIFACT_ROOT.name))
update_session(export_bundle_path=str(bundle_path))
log_event('run_bundle_created', bundle_path=str(bundle_path))
print('\nRun artifacts ready:')
print('-', RUN_ARTIFACT_ROOT)
print('-', EVENT_LOG_PATH)
print('-', SESSION_SUMMARY_PATH)
print('-', bundle_path)

## Next Step

If the model loads cleanly and the smoke test works on the selected GPU, replace the heuristic baseline or toy prompt path with the stronger competition inference loop and build predictions from the competition test input.

Every run now leaves behind:
- `events.jsonl` for stage-by-stage logging,
- `session_summary.json` for the latest structured state,
- preview files and scaffolded submission artifacts,
- and a zip bundle in `/kaggle/working` for export or repo sync.